# **01: Exploratory Data Analysis**

This first notebook contains the full exploratory analysis of the *online_retail_II* dataset, and the relative cleaning process.

In [58]:
import pandas as pd

In [59]:
df = pd.read_csv("../data/online_retail_II.csv")

In [77]:
original_len = len(df)
df.shape

(1067371, 8)

In [61]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.1 Look for null or missing values

In [62]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [63]:
df[df["Description"].isnull()].head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


In [64]:
df[df["Customer ID"].isnull()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom


In [66]:
null_desc = len(df[df["Description"].isnull()])
null_id = len(df[df["Customer ID"].isnull()])
zero_price = len(df[df["Price"]==0])
neg_price = len(df[df["Price"]<0])
print(f"{null_desc} rows with null Description;\n{null_id} rows with null Customer ID;\n{zero_price} rows with Price=0.00;\n{neg_price} rows with negative Price.")

4382 rows with null Description;
243007 rows with null Customer ID;
6202 rows with Price=0.00;
5 rows with negative Price.


In [67]:
# Inspect correlation between zero-price and missing values
all_conditions = len(df[(df["Customer ID"].isnull()) & (df["Description"].isnull()) & (df["Price"] == 0)])
with_description = len(df[(df["Price"]==0) & (df["Customer ID"].isnull())]) - all_conditions
just_zero_price = len(df[(df["Price"]==0) & (df["Description"].notnull()) & (df["Customer ID"].notnull())])

In [68]:
print(f"Of ZERO-PRICED rows:\n{all_conditions} have both Description and Customer ID missng;")
print(f"{with_description} are missing just the Customer ID;")
print(f"{just_zero_price} have both Description and Customer ID.")

Of ZERO-PRICED rows:
4382 have both Description and Customer ID missng;
1749 are missing just the Customer ID;
71 have both Description and Customer ID.


In [69]:
# Zero-priced and no Customer ID
df[(df["Price"]==0) & (df["Customer ID"].isnull()) & (df["Description"].notnull())].head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom


In [71]:
# Just zero-priced
df[(df["Price"]==0) & (df["Description"].notnull()) & (df["Customer ID"].notnull())].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070.0,United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071.0,United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258.0,United Kingdom
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417.0,Belgium
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858.0,United Kingdom


In [86]:
# Just Customer ID Missing
df[(df["Customer ID"].isnull()) & (df["Description"].notnull()) & (df["Price"]>0)].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom
1059,489548,22131,FOOD CONTAINER SET 3 LOVE HEART,2,2009-12-01 12:32:00,1.95,NaN,United Kingdom
1060,489548,22079,RIBBON REEL HEARTS DESIGN,10,2009-12-01 12:32:00,1.65,NaN,United Kingdom
1061,489548,22138,BAKING SET 9 PIECE RETROSPOT,3,2009-12-01 12:32:00,4.95,NaN,United Kingdom
1062,489548,22147,FELTCRAFT BUTTERFLY HEARTS,2,2009-12-01 12:32:00,1.45,NaN,United Kingdom


In [87]:
# Negative price
df[df["Price"]<0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


Upon first inspection we notice a few things:        

1. All rows with missing *Description*, also have null *Customer ID* and zero *Price*: these $4382$ rows may be removed, since too much info is missing for a relevant 
analysis.        

2. $1749$ rows with missing *Customer ID* also have zero price. Upon inspection, it appears that most of these rows refer to damaged or lost products, and even those who have valid Description are already missing too much info (Customer ID and price) so I chose to remove them altogether from further analysis.    

3. The rows in which ONLY *Customer ID* is null are $236876$; they may be kept since they retain meaningful information for statistical analysis on sales. They can be kept out of the analysis if later on we decide to focus on a customer-based analysis.    

4. The rows in which all info is present but the *Price* is set to $0$ are $71$. Of these rows, some have descriptions that point to noise like "Manual" or "test". These may be removed.    

5. $5$ rows have negative price, and they all have the same description: "Adjust bad debt". These can be safely removed as they correspond to accounting adjustments and do not represent actual transactions.

## 1.2 Remove noise

#### **Removing negative-price & missing Description rows**

In [78]:
# Remove negative price rows
clean_df = df[df["Price"]>=0]

# Remove rows with missing Description (also missing Customer ID and having zero-price)
clean_df = clean_df[clean_df["Description"].notnull()]

print(f"Number of data samples expected: {original_len-null_desc-neg_price}\nNumber of data samples: {len(clean_df)}")

Number of data samples expected: 1062984
Number of data samples: 1062984


#### **Removing zero-priced + missing Customer ID rows**

In [79]:
prev_len = len(clean_df) # for checking

# Removing rows with zero price and missing Customer ID
mask = (clean_df["Price"]==0) & (clean_df["Customer ID"].isnull())
clean_df = clean_df[~mask]

print(f"Number of data samples expected: {prev_len-with_description}\nNumber of data samples: {(len(clean_df))}")

Number of data samples expected: 1061235
Number of data samples: 1061235


#### **Removing samples not related to product sales**
This step focuses on the remainder of the data samples. Inspecting rows that only have zero price, we noticed that some rows were to be considered invalid based on their Description (eg. "test¨, "manual",...) or the Stock Code that was differently formatted.     
This is valid not only for zero-priced rows but the whole dataset, so in this section we focus on removing invalid samples from the whole remaining dataset.    
    

Removing rows with specific Descripions may be too hard to implement, since "Manual" and "Test" are just some examples of invalid Descriptions and we cannot afford to look at all of them.    
Most valid samples have all uppergase Description, but I preferred to not use this criterium since it may be too strict (removed all rows where a unit of measurement like grams or cm is present in the description).    
    

The Stock Code may be more indicative.


In [80]:
# Before inspecting Stock Codes and Descriptions, let us clean them up by removing useless blank spaces
clean_df["StockCode"]=clean_df["StockCode"].str.strip()
clean_df["Description"]=clean_df["Description"].str.strip()

##### 1. Non-alphanumeric Stock codes

In [81]:
# Are there non alphanumeric stock codes for valid data samples?
mask = (clean_df["StockCode"].str.isalnum())
print("Unique entries with non alphanumeric Stock Code:")
print(clean_df[~mask][["StockCode", "Description"]].drop_duplicates().sort_values("StockCode"))

Unique entries with non alphanumeric Stock Code:
           StockCode                         Description
18410   BANK CHARGES                        Bank Charges
40904   gift_0001_10  Dotcomgiftshop Gift Voucher £10.00
32048   gift_0001_20  Dotcomgiftshop Gift Voucher £20.00
45212   gift_0001_30  Dotcomgiftshop Gift Voucher £30.00
299652  gift_0001_40  Dotcomgiftshop Gift Voucher £40.00
41263   gift_0001_50  Dotcomgiftshop Gift Voucher £50.00
235291  gift_0001_70  Dotcomgiftshop Gift Voucher £70.00
31079   gift_0001_80  Dotcomgiftshop Gift Voucher £80.00


All non-alphanumeric Stock Codes are not products. Bank charges can be safely removed as they correspond to external charges, while Gift vouchers lie in in a "grey area".
They are real transactions with revenue, but are not specific products.     
**For the scope of this analysis** (which will focus on product-based revenue) they will be removed, even though they would have been relevant for a total revenue analysis. 

In [82]:
# Removing non-alphanumeric Stock Codes
clean_df = clean_df[clean_df["StockCode"].str.isalnum()]

##### 2. Stock codes' lengths

In [83]:
# How are lengths distributed?
clean_df["StockCode"].str.len().value_counts().sort_index()

StockCode
1      1708
2       279
3      1439
4      2150
5    928107
6    125820
7      1384
8       100
9        68
Name: count, dtype: int64

In [84]:
# Explore less common stock code lengths
for i in [1,2,3,4,7,8,9]:
    print(f"\nUnique Objects with StockCode of length {i}:")
    print(clean_df[clean_df["StockCode"].str.len() == i][["StockCode", "Description"]].drop_duplicates().sort_values("StockCode"))


Unique Objects with StockCode of length 1:
       StockCode      Description
825443         B  Adjust bad debt
735            D         Discount
2697           M           Manual
114061         S          SAMPLES
96608          m           Manual

Unique Objects with StockCode of length 2:
     StockCode Description
9292        C2    CARRIAGE

Unique Objects with StockCode of length 3:
     StockCode     Description
2379       DOT  DOTCOM POSTAGE

Unique Objects with StockCode of length 4:
       StockCode                 Description
842969      CRUK             CRUK Commission
62299       PADS  PADS TO MATCH ALL CUSHIONS
89          POST                     POSTAGE

Unique Objects with StockCode of length 7:
       StockCode                         Description
613      15056BL             EDWARDIAN PARASOL BLACK
2620     15056bl             EDWARDIAN PARASOL BLACK
1762     79323GR                 GREEN CHERRY LIGHTS
572      79323LP            LIGHT PINK CHERRY LIGHTS
249672   ADJUST

The great majority of samples has a stock code of length 5-6.   
Of the entries with different StockCode lengths:     
   
**Lenghts that can be removed**: 
Entries with Stock Codes of length 1-3, as they refer either to adjustments or delivery fees and do not serve the purpose of a sales analysis.       
    

**Specific Stock Codes that can be removed**:
For Stock Codes with lenght >= 4, all entries with codes in:\["POST", "CRUK","ADJUST2", "TEST001", "TEST002", "AMAZONFEE"\]  can be safely removed.      
These all refer either to manual adjustments or external fees for deliveries or other types of charges: neither are products and therefore are irrelevant for a sales-based analysis.    

In [85]:
# Removing all entries with len(StockCode)<4
clean_df = clean_df[clean_df["StockCode"].str.len()>=4]

# Removing all entries with specific codes
non_prod = ["POST", "CRUK","ADJUST2", "TEST001", "TEST002", "AMAZONFEE"]
mask = clean_df["StockCode"].isin(non_prod)
clean_df = clean_df[~mask]


## 1.3 Final clean-up

#### **Check for duplicates**
Let us now make sure that there are no duplicates in the entries

In [87]:
print(clean_df.duplicated().sum())
clean_df[clean_df.duplicated(keep=False)].sort_values(by=["Invoice", "StockCode"]).head(10)

34036


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [88]:
clean_df = clean_df.drop_duplicates()

#### **Check column types**
It's now best to make sure that InvoiceDates are in the datetime format, and that Customer ID is an int for clarity

In [89]:
clean_df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [90]:
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"])
clean_df["Customer ID"] = clean_df["Customer ID"].astype("Int64")
clean_df.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object

#### **Check for returns and create a separate df**
Returns may be stored as entries with negative quantity. It may be useful best to additionally create separate dataframes for sales and returns for clarity.

In [91]:
len(clean_df[clean_df["Quantity"]<0])

17946

In [92]:
df_sales = clean_df[clean_df["Quantity"]>0].copy()
df_returns = clean_df[clean_df["Quantity"]<0].copy()

#df.to_csv("../data/df_clean.csv", index=False)
df_sales.to_csv("../data/sales_clean.csv", index=False)
df_returns.to_csv("../data/returns_clean.csv", index=False)

## 1.4 Summary

In [98]:
remov = original_len-len(clean_df)

print(f"Original dataset: {original_len} rows")
print(f"After cleaning: {len(clean_df)} rows")
print(f"Sales records: {len(df_sales)}")
print(f"Return records: {len(df_returns)}")
print(f"Removed: {remov} rows ({(remov/original_len)*100:.4f}% of original)")

Original dataset: 1067371 rows
After cleaning: 1021400 rows
Sales records: 1003454
Return records: 17946
Removed: 45971 rows (4.3069% of original)
